# 🗡️ Hey Aragorn Wake Word Training

Train a custom wake word model for "Hey Aragorn" using micro-wake-word.

**Run each cell in order.**

## Step 1: Check GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 2: Install Dependencies (Nuclear Option)

In [ ]:
# Nuclear option: uninstall numpy/scipy first to avoid conflicts,
# then clean install everything
print("\u2622\ufe0f Nuclear option: removing numpy/scipy first...")
!pip uninstall -y numpy scipy 2>/dev/null

print("\nRemoving other conflicting packages...")
!pip uninstall -y jax jaxlib tensorstore tensorflow-decision-forests tf-keras tensorflow-text opencv-python opencv-python-headless opencv-contrib-python shap ydf grain pytensor xarray-einstats rasterio tobler cupy-cuda12x 2>/dev/null

print("\nInstalling compatible versions...")
!pip install --quiet numpy==1.26.4 scipy 2>/dev/null
!pip install --quiet tensorflow==2.16.2 protobuf==4.25.8 ml-dtypes==0.3.2 2>/dev/null
!pip install --quiet piper-tts piper-phonemize-cross onnxruntime --no-deps 2>/dev/null
!pip install --quiet pyyaml datasets mmap-ninja tqdm audiomentations webrtcvad-wheels 2>/dev/null
print("\n\u2705 Dependencies installed!")
print("\n\u26a0\ufe0f IMPORTANT: Restart the runtime now!")
print("   Runtime \u2192 Restart session (Ctrl+M .)")
print("   Then continue from Step 3")

## ⚠️ Restart Required

**You must restart the runtime before continuing!**

Click: **Runtime → Restart session** (or press Ctrl+M .)

Then continue from Step 3.

## Step 3: Clone Repositories

In [ ]:
import subprocess
import os
import sys

if not os.path.exists("microWakeWord"):
    subprocess.run(["git", "clone", "https://github.com/kahrendt/microWakeWord.git"], check=True)
    subprocess.run(["pip", "install", "-q", "-e", "microWakeWord"], check=True)

if not os.path.exists("piper-sample-generator"):
    subprocess.run(["git", "clone", "https://github.com/rhasspy/piper-sample-generator.git"], check=True)

print("\u2705 Ready!")

## Step 4: Download Piper Voice Model

In [ ]:
import os
import urllib.request

os.makedirs("piper-sample-generator/models", exist_ok=True)

url = "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt"
path = "piper-sample-generator/models/en_US-libritts_r-medium.pt"

if not os.path.exists(path):
    print("Downloading...")
    urllib.request.urlretrieve(url, path)
    print("\u2705 Done!")
else:
    print("\u2705 Already exists")

## Step 5: Configure

In [ ]:
TARGET_WORD = "hey_air_uh_gorn"
NUM_SAMPLES = 1000
TRAINING_STEPS = 10000

print(f"Word: {TARGET_WORD}")
print(f"Samples: {NUM_SAMPLES}")
print(f"Steps: {TRAINING_STEPS}")

## Step 6: Generate Test Sample

Generate just 1 sample to verify the phonetic spelling sounds right.

**Listen to the audio below.** If it doesn't sound like your intended wake word, go back to Step 5 and adjust the phonetic spelling.

Common phonetic tips:
- Use underscores between syllables: `hey_air_uh_gorn`
- For sounds like 'sh', 'ch', 'th': use `sh`, `ch`, `th`
- For long vowels: add `ee` or `oo` (e.g., `see` for long 'e')
- Experiment until it sounds right!

In [ ]:
import subprocess
import os
import sys
from IPython.display import Audio, display

os.makedirs("generated_samples", exist_ok=True)

print("Generating 1 test sample to verify pronunciation...")

# Add repo to path so piper_train can be found
if 'piper-sample-generator' not in sys.path:
    sys.path.insert(0, 'piper-sample-generator')

result = subprocess.run([
    sys.executable,
    '-m',
    'piper_sample_generator',
    TARGET_WORD,
    '--max-samples', '1',
    '--batch-size', '1',
    '--model', 'piper-sample-generator/models/en_US-libritts_r-medium.pt',
    '--output-dir', 'generated_samples',
], capture_output=True, text=True)

if result.returncode == 0:
    sample_files = sorted([f for f in os.listdir("generated_samples") if f.endswith('.wav')])
    if sample_files:
        sample_path = os.path.join("generated_samples", sample_files[0])
        print(f"\nPlaying: {sample_files[0]}")
        print(f"Phonetic spelling used: '{TARGET_WORD}'")
        print("\nDoes this sound like your intended wake word?")
        print("If not, go back to Step 5 and adjust the spelling.\n")
        display(Audio(sample_path))
    else:
        print("No samples found in output directory.")
else:
    print("Error:", result.stderr)

## Step 7: Generate All Samples

In [ ]:
import subprocess
import os
import sys

print(f"Generating {NUM_SAMPLES} samples...")

# Add repo to path so piper_train can be found
if 'piper-sample-generator' not in sys.path:
    sys.path.insert(0, 'piper-sample-generator')

result = subprocess.run([
    sys.executable,
    '-m',
    'piper_sample_generator',
    TARGET_WORD,
    '--max-samples', str(NUM_SAMPLES),
    '--batch-size', '100',
    '--model', 'piper-sample-generator/models/en_US-libritts_r-medium.pt',
    '--output-dir', 'generated_samples',
], capture_output=True, text=True)

if result.returncode == 0:
    count = len([f for f in os.listdir('generated_samples') if f.endswith('.wav')])
    print(f"\u2705 Generated {count} samples!")
else:
    print("Error:", result.stderr)

## Step 8: Download Augmentation Data

In [ ]:
import os

print("Downloading augmentation data...")
print("This is ~1.3GB and may take 5-10 minutes...")

os.makedirs("mit_rirs", exist_ok=True)
if not os.listdir("mit_rirs"):
    print("Downloading from OpenSLR...")
    !wget --progress=bar:force -O /tmp/rirs_noises.zip https://www.openslr.org/resources/28/rirs_noises.zip
    print("Extracting...")
    !unzip -q /tmp/rirs_noises.zip -d mit_rirs
    print("\u2713 RIRs ready!")
else:
    print("\u2713 RIRs already downloaded")

os.makedirs("audioset_16k", exist_ok=True)
os.makedirs("fma_16k", exist_ok=True)

print("\u2705 Done!")

## Step 9: Generate Spectrograms

In [ ]:
import os
import sys

if 'microWakeWord' not in sys.path:
    sys.path.insert(0, 'microWakeWord')

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap

print("Generating spectrograms...")

clips = Clips(
    input_directory="generated_samples",
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        "SevenBandParametricEQ": 0.1,
        "TanhDistortion": 0.1,
        "PitchShift": 0.1,
        "BandStopFilter": 0.1,
        "AddColorNoise": 0.1,
        "AddBackgroundNoise": 0.75,
        "Gain": 1.0,
        "RIR": 0.5,
    },
    impulse_paths=["mit_rirs"],
    background_paths=["fma_16k", "audioset_16k"],
    background_min_snr_db=-5,
    background_max_snr_db=10,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

os.makedirs("generated_augmented_features/training", exist_ok=True)

spectrograms = SpectrogramGeneration(
    clips=clips,
    augmenter=augmenter,
    slide_frames=10,
    step_ms=10,
)

RaggedMmap.from_generator(
    out_dir="generated_augmented_features/training/wakeword_mmap",
    sample_generator=spectrograms.spectrogram_generator(split="train", repeat=2),
    batch_size=100,
    verbose=True,
)

print("\u2705 Done!")

## Step 10: Download Negative Datasets

In [ ]:
import os
import subprocess
import urllib.request

print("Downloading negative datasets...")

os.makedirs("negative_datasets", exist_ok=True)

base_url = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
files = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']

for fname in files:
    zip_path = f"negative_datasets/{fname}"
    if not os.path.exists(zip_path.replace('.zip', '')):
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(base_url + fname, zip_path)
        subprocess.run(["unzip", "-q", zip_path, "-d", "negative_datasets"], check=True)
        os.remove(zip_path)

print("\u2705 Done!")

## Step 11: Create Config

In [ ]:
import yaml

config = {
    "window_step_ms": 10,
    "train_dir": "trained_models/wakeword",
    "spectrogram_length": 204,
    "stride": 3,
    "features": [
        {"features_dir": "generated_augmented_features", "sampling_weight": 2.0, "penalty_weight": 1.0, "truth": True, "truncation_strategy": "truncate_start", "type": "mmap"},
        {"features_dir": "negative_datasets/speech", "sampling_weight": 10.0, "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"},
        {"features_dir": "negative_datasets/dinner_party", "sampling_weight": 10.0, "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"},
        {"features_dir": "negative_datasets/no_speech", "sampling_weight": 5.0, "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"},
        {"features_dir": "negative_datasets/dinner_party_eval", "sampling_weight": 0.0, "penalty_weight": 1.0, "truth": False, "truncation_strategy": "split", "type": "mmap"},
    ],
    "training_steps": [TRAINING_STEPS],
    "positive_class_weight": [1],
    "negative_class_weight": [20],
    "learning_rates": [0.001],
    "batch_size": 128,
    "eval_step_interval": 500,
    "clip_duration_ms": 1500,
    "target_minimization": 0.9,
    "minimization_metric": None,
    "maximization_metric": "average_viable_recall",
}

with open("training_parameters.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("\u2705 Config saved!")

## Step 12: Train Model

This takes 1-3 hours. Go get coffee!

In [ ]:
import subprocess
import sys

print("\ud83d\ude80 Starting training...")
print(f"\u23f3 ~{TRAINING_STEPS // 10000} hour(s)...")

result = subprocess.run([
    sys.executable, "-m", "microwakeword.model_train_eval",
    "--training_config=training_parameters.yaml",
    "--train=1",
    "--restore_checkpoint", "1",
    "--test_tf_nonstreaming", "0",
    "--test_tflite_nonstreaming", "0",
    "--test_tflite_nonstreaming_quantized", "0",
    "--test_tflite_streaming", "0",
    "--test_tflite_streaming_quantized", "1",
    "mixednet",
    "--pointwise_filters", "64,64,64,64",
    "--repeat_in_block", "1, 1, 1, 1",
    "--mixconv_kernel_sizes", "'[5], [7,11], [9,15], [23]'",
    "--residual_connection", "0,0,0,0",
    "--first_conv_filters", "32",
    "--first_conv_kernel_size", "5",
    "--stride", "3",
])

if result.returncode == 0:
    print("\u2705 Training complete!")
else:
    print("\u274c Failed")

## Step 13: Download Model

In [ ]:
import os
from google.colab import files

model_path = "trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"

if os.path.exists(model_path):
    size_kb = os.path.getsize(model_path) / 1024
    print(f"\u2705 Model: {size_kb:.1f} KB")
    files.download(model_path)
    print("\n\ud83c\udf89 Done! Check your downloads.")
else:
    print("\u274c Model not found")